In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

RANDOM_STATE = 42
TEST_SIZE = 0.2

C:\Users\ahmed\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Load cleaned dataset
df = pd.read_csv("../data/cleaned_spam.csv")

# Remove missing values
df = df.dropna(subset=["clean_message", "label"]).reset_index(drop=True)

# Features and labels
X = df["clean_message"].astype(str)
y = df["label"].map({"ham": 0, "spam": 1})

# Same split used by the team
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Dataset shape:", df.shape)
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("\nTraining class distribution:")
print(y_train.value_counts())

Dataset shape: (5167, 3)
Training samples: 4133
Test samples: 1034

Training class distribution:
label
0    3611
1     522
Name: count, dtype: int64


In [3]:
# Baseline: CountVectorizer + Multinomial Naive Bayes

baseline_vectorizer = CountVectorizer()

X_train_count = baseline_vectorizer.fit_transform(X_train)
X_test_count = baseline_vectorizer.transform(X_test)

baseline_model = MultinomialNB()
baseline_model.fit(X_train_count, y_train)

baseline_pred = baseline_model.predict(X_test_count)

print("Baseline model trained successfully!")
print("Training features:", X_train_count.shape)
print("Test features:", X_test_count.shape)

Baseline model trained successfully!
Training features: (4133, 7632)
Test features: (1034, 7632)


In [4]:
# Hyperparameter Tuning - Multinomial Naive Bayes

param_grid = {
    "alpha": [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
}

grid_search = GridSearchCV(
    estimator=MultinomialNB(),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train_count, y_train)

print("Best Alpha:", grid_search.best_params_["alpha"])
print("Best CV F1:", round(grid_search.best_score_, 4))

Best Alpha: 1.0
Best CV F1: 0.9187


In [5]:
# Feature Optimization - CountVectorizer

vectorizer_param_grid = {
    "ngram_range": [(1, 1), (1, 2)],
    "min_df": [1, 2, 3],
    "max_df": [0.95, 1.0]
}

best_feature_score = 0
best_feature_params = None
best_vectorizer = None
best_model = None

for params in [
    {
        "ngram_range": ngram_range,
        "min_df": min_df,
        "max_df": max_df
    }
    for ngram_range in vectorizer_param_grid["ngram_range"]
    for min_df in vectorizer_param_grid["min_df"]
    for max_df in vectorizer_param_grid["max_df"]
]:
    
    vectorizer = CountVectorizer(
        ngram_range=params["ngram_range"],
        min_df=params["min_df"],
        max_df=params["max_df"]
    )
    
    X_train_features = vectorizer.fit_transform(X_train)
    
    model = MultinomialNB(alpha=1.0)
    
    grid = GridSearchCV(
        model,
        {"alpha": [0.5, 1.0, 1.5]},
        scoring="f1",
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train_features, y_train)
    
    if grid.best_score_ > best_feature_score:
        best_feature_score = grid.best_score_
        best_feature_params = params
        best_vectorizer = vectorizer
        best_model = grid.best_estimator_

print("Best Feature Parameters:")
print(best_feature_params)

print("\nBest Alpha:")
print(best_model.alpha)

print("\nBest CV F1:")
print(round(best_feature_score, 4))

Best Feature Parameters:
{'ngram_range': (1, 1), 'min_df': 3, 'max_df': 0.95}

Best Alpha:
0.5

Best CV F1:
0.9365


In [6]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Transform test data using the BEST vectorizer
X_test_best = best_vectorizer.transform(X_test)

# Predict using the BEST tuned model
y_pred_tuned = best_model.predict(X_test_best)

# Calculate test metrics
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
tuned_precision = precision_score(y_test, y_pred_tuned)
tuned_recall = recall_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)

print("TUNED MODEL - FINAL TEST RESULTS")
print("=" * 45)
print(f"Accuracy :  {tuned_accuracy:.4f}")
print(f"Precision:  {tuned_precision:.4f}")
print(f"Recall   :  {tuned_recall:.4f}")
print(f"F1-score :  {tuned_f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

TUNED MODEL - FINAL TEST RESULTS
Accuracy :  0.9865
Precision:  0.9680
Recall   :  0.9237
F1-score :  0.9453

Confusion Matrix:
[[899   4]
 [ 10 121]]


In [11]:
from sklearn.svm import LinearSVC

# LinearSVC Hyperparameter Tuning
svc_param_grid = {
    "C": [0.1, 0.5, 1.0, 2.0, 5.0],
    "class_weight": [None, "balanced"]
}

svc_grid = GridSearchCV(
    estimator=LinearSVC(
        random_state=RANDOM_STATE,
        max_iter=5000
    ),
    param_grid=svc_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

# IMPORTANT: tuning uses training data only
svc_grid.fit(X_train_count, y_train)

print("Best LinearSVC Parameters:")
print(svc_grid.best_params_)

print("\nBest CV F1:")
print(round(svc_grid.best_score_, 4))

Best LinearSVC Parameters:
{'C': 0.5, 'class_weight': 'balanced'}

Best CV F1:
0.9273


In [13]:
# Prepare TF-IDF features

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF features prepared successfully!")
print("Training features:", X_train_tfidf.shape)
print("Test features:", X_test_tfidf.shape)

TF-IDF features prepared successfully!
Training features: (4133, 7632)
Test features: (1034, 7632)


In [14]:
# LinearSVC with TF-IDF - Hyperparameter Tuning

svc_tfidf_grid = GridSearchCV(
    estimator=LinearSVC(
        random_state=RANDOM_STATE,
        max_iter=5000
    ),
    param_grid={
        "C": [0.1, 0.5, 1.0, 2.0, 5.0],
        "class_weight": [None, "balanced"]
    },
    scoring="f1",
    cv=5,
    n_jobs=-1
)

svc_tfidf_grid.fit(X_train_tfidf, y_train)

print("Best TF-IDF + LinearSVC Parameters:")
print(svc_tfidf_grid.best_params_)

print("\nBest CV F1:")
print(round(svc_tfidf_grid.best_score_, 4))

Best TF-IDF + LinearSVC Parameters:
{'C': 1.0, 'class_weight': 'balanced'}

Best CV F1:
0.9232


In [15]:
# Feature Optimization: Unigrams + Bigrams with LinearSVC

count_bigram_vectorizer = CountVectorizer(
    ngram_range=(1, 2)
)

X_train_bigram = count_bigram_vectorizer.fit_transform(X_train)
X_test_bigram = count_bigram_vectorizer.transform(X_test)

print("Bigram features prepared!")
print("Training features:", X_train_bigram.shape)
print("Test features:", X_test_bigram.shape)

Bigram features prepared!
Training features: (4133, 42410)
Test features: (1034, 42410)


In [16]:
# LinearSVC tuning with Unigrams + Bigrams

svc_bigram_grid = GridSearchCV(
    estimator=LinearSVC(
        random_state=RANDOM_STATE,
        max_iter=5000
    ),
    param_grid={
        "C": [0.1, 0.5, 1.0, 2.0, 5.0],
        "class_weight": [None, "balanced"]
    },
    scoring="f1",
    cv=5,
    n_jobs=-1
)

svc_bigram_grid.fit(X_train_bigram, y_train)

print("Best Bigram + LinearSVC Parameters:")
print(svc_bigram_grid.best_params_)

print("\nBest CV F1:")
print(round(svc_bigram_grid.best_score_, 4))

Best Bigram + LinearSVC Parameters:
{'C': 0.1, 'class_weight': 'balanced'}

Best CV F1:
0.9153


In [17]:
# Final Best Model
# CountVectorizer + Multinomial Naive Bayes

final_vectorizer = CountVectorizer()

X_train_final = final_vectorizer.fit_transform(X_train)
X_test_final = final_vectorizer.transform(X_test)

final_model = MultinomialNB(alpha=0.5)

final_model.fit(X_train_final, y_train)

print("Final model trained successfully!")
print("Training features:", X_train_final.shape)
print("Test features:", X_test_final.shape)

Final model trained successfully!
Training features: (4133, 7632)
Test features: (1034, 7632)


In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Final predictions on the untouched test set
y_pred_final = final_model.predict(X_test_final)

# Final metrics
final_accuracy = accuracy_score(y_test, y_pred_final)
final_precision = precision_score(y_test, y_pred_final)
final_recall = recall_score(y_test, y_pred_final)
final_f1 = f1_score(y_test, y_pred_final)

print("FINAL MODEL - TEST RESULTS")
print("=" * 45)
print(f"Accuracy :  {final_accuracy:.4f}")
print(f"Precision:  {final_precision:.4f}")
print(f"Recall   :  {final_recall:.4f}")
print(f"F1-score :  {final_f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_final,
    target_names=["ham", "spam"]
))

FINAL MODEL - TEST RESULTS
Accuracy :  0.9884
Precision:  0.9760
Recall   :  0.9313
F1-score :  0.9531

Confusion Matrix:
[[900   3]
 [  9 122]]

Classification Report:
              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       903
        spam       0.98      0.93      0.95       131

    accuracy                           0.99      1034
   macro avg       0.98      0.96      0.97      1034
weighted avg       0.99      0.99      0.99      1034



In [19]:
# Save final model and vectorizer

joblib.dump(final_model, "../models/final_spam_model.pkl")
joblib.dump(final_vectorizer, "../models/final_count_vectorizer.pkl")

print("Final model saved successfully!")
print("Final vectorizer saved successfully!")

Final model saved successfully!
Final vectorizer saved successfully!
